# Transformer / LLM Architecture — Hands-On

**LLM Engineering · Domain 2 · Roadmap Weeks 08/10**

Companion to `02 Literature Notes/LLM Engineering/Transformer Architecture`. Pure numpy forward pass, runs offline.

## 0. Setup — norms, activations, attention

In [ ]:
%pip install -q numpy
import numpy as np
rng = np.random.RandomState(0)
def softmax(x, ax=-1):
    x = x - x.max(ax, keepdims=True); e = np.exp(x); return e / e.sum(ax, keepdims=True)
def layernorm(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdims=True); var = x.var(-1, keepdims=True)
    return g * (x - mu) / np.sqrt(var + eps) + b
def gelu(x): return 0.5*x*(1+np.tanh(0.797885*(x+0.044715*x**3)))
def attn(x, Wqkv, Wo, causal=True):
    seq, d = x.shape
    Q, K, V = (x @ Wqkv).reshape(seq, 3, d).transpose(1,0,2)
    s = Q @ K.T / np.sqrt(d)
    if causal: s = np.where(np.triu(np.ones((seq,seq)),1).astype(bool), -1e9, s)
    return (softmax(s) @ V) @ Wo
print("ok")

## 1. One pre-norm block, stacked N times

In [ ]:
def block(x, p):
    x = x + attn(layernorm(x, p['g1'], p['b1']), p['Wqkv'], p['Wo'])
    h = gelu(layernorm(x, p['g2'], p['b2']) @ p['Wup'])
    return x + h @ p['Wdown']

d, seq, ff = 16, 6, 64
def mkparams():
    return dict(g1=np.ones(d), b1=np.zeros(d), g2=np.ones(d), b2=np.zeros(d),
                Wqkv=rng.randn(d,3*d)*.1, Wo=rng.randn(d,d)*.1,
                Wup=rng.randn(d,ff)*.1, Wdown=rng.randn(ff,d)*.1)
x = rng.randn(seq, d)
layers = [mkparams() for _ in range(6)]
for p in layers: x = block(x, p)
print("after 6 blocks:", x.shape)

## 2. The residual stream keeps information flowing
Zero out a block's sublayer contributions and confirm the input still passes through.

In [ ]:
p = mkparams()
x = rng.randn(seq, d)
# a 'no-op' block (weights ~0) should approximately preserve x via the residual
p0 = {k: (v*0 if k.startswith('W') else v) for k, v in p.items()}
out = block(x, p0)
print("max |out - x| with zeroed sublayers:", np.abs(out - x).max().round(4),
      "-> residual passes input through")

## 3. Parameter counting and the attention/MLP split

In [ ]:
def params(n_layers, d_model, vocab, ff_mult=4):
    attn = 4*d_model*d_model
    mlp = 2*ff_mult*d_model*d_model
    per = attn + mlp
    return n_layers*per + vocab*d_model, attn/per, mlp/per
for name,(L,d_,v) in {"GPT-2 small":(12,768,50257),"GPT-2 XL":(48,1600,50257),"7B-ish":(32,4096,32000)}.items():
    t,a,m = params(L,d_,v)
    print(f"{name:<12} ~{t/1e6:>8.1f}M  attn={a:.0%} mlp={m:.0%} per block")

> Note the MLP holds ~2/3 of each block's parameters — that's where knowledge concentrates.

## 4. Chinchilla intuition: tokens per parameter

In [ ]:
for params_b, tokens_t in [(7,2.0),(13,0.3),(70,1.4)]:
    ratio = tokens_t*1e12 / (params_b*1e9)
    verdict = "well-trained" if ratio >= 15 else "UNDER-trained"
    print(f"{params_b:>3}B params, {tokens_t}T tokens -> {ratio:5.1f} tok/param  {verdict}")

## 5. Exercises
1. Add weight tying: use the same matrix for embedding and unembedding.
2. Swap LayerNorm for RMSNorm (drop the mean subtraction) and compare outputs.
3. Increase ff_mult and watch the parameter split shift toward the MLP.
4. Stack 24 blocks and check activations don't blow up (residual + norm stability).

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Transformer Architecture`
- Snippets: `04 Code Snippets/LLM/A Transformer Block Forward Pass`, `.../Counting Transformer Parameters`
- MOC: `06 Maps of Content/LLM Engineering Concepts`